## Embeddings

In [2]:
!pip install datasets chromadb -q

^C


In [ ]:
!pip install onnxruntime-gpu -q

^C


In [3]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

c:\Users\danie\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danie\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

## 1. Load Dataset

In [4]:
imdb_train_pd = imdb["train"].to_pandas()
imdb_test_pd = imdb["test"].to_pandas()
imdb_train_pd

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [5]:
# getting 1000 for faster processing
samples_per_label = 10000 // imdb_train_pd['label'].nunique()
df_small_train_sample = imdb_train_pd.groupby('label').apply(lambda x: x.sample(samples_per_label)).reset_index(drop=True)
df_small_train_sample

C:\Users\danie\AppData\Local\Temp\ipykernel_18600\1074002351.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small_train_sample = imdb_train_pd.groupby('label').apply(lambda x: x.sample(samples_per_label)).reset_index(drop=True)


,text,label
0,Spoiler warning.<br /><br />When the main char...,0
1,"Judging by some of the comments in IMDB, I was...",0
2,"A truly disturbed, cannibalistic psychopath, J...",0
3,I noticed with some amusement that in the end ...,0
4,I find it heart-warming and inspiring that the...,0
...,...,...
9995,"This is one of those feel good, Saturday after...",1
9996,This movie contains personalities that so deli...,1
9997,"Fred Astaire and Ginger Rogers, Hollywood's pr...",1
9998,"As far as parody films go, there are few that ...",1


## 2. Convert Text To Embeddings

In [6]:
df = df_small_train_sample.copy()

In [7]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [ ]:
# Colab
# from google.colab import userdata

# api_key_hf = userdata.get('HF_TOKEN')

In [8]:
from chromadb.utils import embedding_functions
import os
from dotenv import load_dotenv

load_dotenv()

# Hugging Face embedding function setup
huggingface_emb = embedding_functions.HuggingFaceEmbeddingFunction(
                                                                api_key=os.environ.get("HUGGINGFACEHUB_API_TOKEN"), #local
                                                                #api_key=api_key_hf, #colab
                                                                model_name="sentence-transformers/all-MiniLM-L6-v2",
                                                                )

In [9]:
def generate_embeddings(text):
    return huggingface_emb([text])[0]

In [11]:
df['embeddings'] = df['text'].apply(generate_embeddings)
df

ConnectError: [Errno 11001] getaddrinfo failed

In [ ]:
# from tokenizers import Tokenizer
# import onnxruntime as ort
# import numpy as np
# from typing import List

# # Use pytorches default epsilon for division by zero
# # https://pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html
# def normalize(v):
#     norm = np.linalg.norm(v, axis=1)
#     norm[norm == 0] = 1e-12
#     return v / norm[:, np.newaxis]

# # Sampel implementation of the default sentence-transformers model using ONNX
# class DefaultEmbeddingModel():

#     def __init__(self):
#         # max_seq_length = 256, for some reason sentence-transformers uses 256 even though the HF config has a max length of 128
#         # https://github.com/UKPLab/sentence-transformers/blob/3e1929fddef16df94f8bc6e3b10598a98f46e62d/docs/_static/html/models_en_sentence_embeddings.html#LL480
#         self.tokenizer = Tokenizer.from_file("onnx/tokenizer.json")
#         self.tokenizer.enable_truncation(max_length=256)
#         self.tokenizer.enable_padding(pad_id=0, pad_token="[PAD]", length=256)
#         self.model = ort.InferenceSession("onnx/model.onnx")


#     def __call__(self, documents: List[str], batch_size: int = 32):
#         all_embeddings = []

#         for i in range(0, len(documents), batch_size):
#             batch = documents[i:i + batch_size]
#             encoded = [self.tokenizer.encode(d) for d in batch]
#             input_ids = np.array([e.ids for e in encoded])
#             attention_mask = np.array([e.attention_mask for e in encoded])
#             onnx_input = {
#                         "input_ids": np.array(input_ids, dtype=np.int64),
#                         "attention_mask": np.array(attention_mask, dtype=np.int64),
#                         "token_type_ids": np.array([np.zeros(len(e), dtype=np.int64) for e in input_ids], dtype=np.int64),
#                         }
#             model_output = self.model.run(None, onnx_input)
#             last_hidden_state = model_output[0]
#             # Perform mean pooling with attention weighting
#             input_mask_expanded = np.broadcast_to(np.expand_dims(attention_mask, -1), last_hidden_state.shape)
#             embeddings = np.sum(last_hidden_state * input_mask_expanded, 1) / np.clip(input_mask_expanded.sum(1), a_min=1e-9, a_max=None)
#             embeddings = normalize(embeddings).astype(np.float32)
#             all_embeddings.append(embeddings)

#         return np.concatenate(all_embeddings)

In [ ]:
# model = DefaultEmbeddingModel()

# def generate_embeddings(text):
#     return model([text])[0]

In [ ]:
# %%time

# df['embeddings'] = df['text'].apply(generate_embeddings)
# df

CPU times: total: 26min 40s
Wall time: 14min 43s


,text,label,embeddings
0,"OK, I really don't have too much to say about ...",0,"[-0.088317834, 0.03389106, -0.06940427, -0.082..."
1,(This is a review of the later English release...,0,"[-0.013487424, -0.040132232, 0.039071724, -0.0..."
2,"If I could give it a zero, I'd change my mind ...",0,"[0.00069093273, -0.0055938736, -0.00545048, -0..."
3,"Wowwwwww, about an hour ago I finally finished...",0,"[-0.10546522, -0.060921054, 0.029692814, 0.000..."
4,The earlier part of the film was rather enjoya...,0,"[0.005623572, -0.039065428, -0.039952625, -0.0..."
...,...,...,...
9995,The story centers around Barry McKenzie who mu...,1,"[0.0287751, -0.0010590936, -0.020167436, -0.01..."
9996,Bruce Almighty is the best Jim Carrey work sin...,1,"[-0.08107941, -0.019554712, -0.003819812, 0.00..."
9997,"Based on Robert Louis Stevenson's St. Ives, th...",1,"[-0.06544652, 0.015919888, -0.03437675, -0.046..."
9998,I rented this movie from blockbuster on a whim...,1,"[-0.041031826, -0.020044593, 0.003453552, -0.0..."


## 3. Let's Classify Using Sklearn on Embeddings

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

X = df["embeddings"].to_list()
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=667,
                                                    )


knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy}')

Accuracy: 0.761
